In [1]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import argparse
import os
import sys
import time

In [3]:
import os, sys
if sys.platform == "win32":
    import ctypes
    ABOVE_NORMAL_PRIORITY = 0x8000
    ctypes.windll.kernel32.SetPriorityClass(
        ctypes.windll.kernel32.GetCurrentProcess(),
        ABOVE_NORMAL_PRIORITY
    )

In [4]:
# sys.path.insert(0, os.path.dirname(__file__))

from config    import AUDIO_DIR, LIPSYNC_DIR, OUTPUT_DIR
from script_parser import load_script, get_all_lines
from voice_gen     import generate_all_voices
from lip_sync      import run_all_lipsync
from timeline      import build_timeline, get_total_duration
from audio_mixer   import build_master_audio
from renderer      import render_video
# from parallel_renderer import render_video_parallel as render_video

In [5]:
script_path = "C:\\Users\\Arunabh\\Desktop\\YouTube\\Channel_1_01\\stickman_animation\\assets\\scripts\\test_tiny.json"
script = load_script(script_path)

In [6]:
# script

In [7]:
lines  = get_all_lines(script)

In [8]:
print(f"      {len(script['scenes'])} scene(s), {len(lines)} dialogue lines")
print(f"      Characters: {', '.join(script['characters'].keys())}")

      2 scene(s), 6 dialogue lines
      Characters: alice, bob, charlie, diana


In [9]:
audio_map = generate_all_voices(lines, verbose=True)

Generating voices:   0%|          | 0/6 [00:00<?, ?it/s]

Generating voices: 100%|██████████| 6/6 [00:00<00:00, 71.77it/s]

assets/audio\s001_l001.wav
assets/audio\s001_l002.wav
assets/audio\s001_l003.wav
assets/audio\s002_l001.wav
assets/audio\s002_l002.wav
assets/audio\s002_l003.wav


In [10]:
# from elevenlabs.client import ElevenLabs

# client = ElevenLabs(
#     api_key="sk_9ddaddfca553dc6df32e3aee0a5fe3a5ebe5f1b0c592dee4"
# )

# voices = client.voices.get_all()

# for v in voices.voices:
#     print(v.name, v.voice_id, v.category)

In [11]:
print("\n[3/6] Running lip sync (Rhubarb)...")
lipsync_map = run_all_lipsync(lines, audio_map, verbose=True)


[3/6] Running lip sync (Rhubarb)...


Running lip sync: 100%|██████████| 6/6 [00:00<00:00, 83.67it/s]


In [12]:
# ── Step 4: Build Timeline ───────────────────────────────────────────────
print("\n[4/6] Building master timeline...")
timeline       = build_timeline(script, audio_map, lipsync_map)
total_duration = get_total_duration(timeline)
print(f"      Total video duration: {total_duration:.1f}s  "
        f"({total_duration / 60:.1f} minutes)")


[4/6] Building master timeline...
      Total video duration: 29.6s  (0.5 minutes)


In [13]:
# ── Step 5: Mix Audio ────────────────────────────────────────────────────
print("\n[5/6] Mixing master audio...")
script_name  = os.path.splitext(os.path.basename(script_path))[0]
audio_out    = os.path.join(OUTPUT_DIR, f"{script_name}_audio.wav")
os.makedirs(OUTPUT_DIR, exist_ok=True)
build_master_audio(timeline, audio_out)


[5/6] Mixing master audio...
✓ Master audio saved: assets/output\test_tiny_audio.wav  (29.6s)


'assets/output\\test_tiny_audio.wav'

In [14]:
# ── Step 6: Render Video ─────────────────────────────────────────────────
print("\n[6/6] Rendering video...")
video_out = os.path.join(OUTPUT_DIR, f"{script_name}.mp4")
render_video(timeline, script, audio_out, video_out, verbose=True)


[6/6] Rendering video...
Rendering 709 frames  (29.6s @ 24fps)
Output: assets/output\test_tiny.mp4


Rendering frames: 100%|██████████| 709/709 [00:07<00:00, 88.92it/s]



✓ Video saved: assets/output\test_tiny.mp4


In [ ]:
elapsed = time.time() - start
print("\n" + "=" * 60)
print(f"  DONE  ({elapsed:.0f}s total)")
print(f"  Output: {video_out}")
print("=" * 60)

In [ ]:
def run_pipeline(script_path: str):
    start = time.time()
    print("=" * 60)
    print("  STICKMAN CARTOON PIPELINE")
    print("=" * 60)

    # ── Step 1: Parse Script ─────────────────────────────────────────────────
    print("\n[1/6] Loading script...")
    script = load_script(script_path)
    lines  = get_all_lines(script)
    print(f"      {len(script['scenes'])} scene(s), {len(lines)} dialogue lines")
    print(f"      Characters: {', '.join(script['characters'].keys())}")

    # ── Step 2: Generate Voices ──────────────────────────────────────────────
    print("\n[2/6] Generating voices (ElevenLabs)...")
    audio_map = generate_all_voices(lines, verbose=True)

    # ── Step 3: Lip Sync ─────────────────────────────────────────────────────
    print("\n[3/6] Running lip sync (Rhubarb)...")
    lipsync_map = run_all_lipsync(lines, audio_map, verbose=True)

    # ── Step 4: Build Timeline ───────────────────────────────────────────────
    print("\n[4/6] Building master timeline...")
    timeline       = build_timeline(script, audio_map, lipsync_map)
    total_duration = get_total_duration(timeline)
    print(f"      Total video duration: {total_duration:.1f}s  "
          f"({total_duration / 60:.1f} minutes)")

    # ── Step 5: Mix Audio ────────────────────────────────────────────────────
    print("\n[5/6] Mixing master audio...")
    script_name  = os.path.splitext(os.path.basename(script_path))[0]
    audio_out    = os.path.join(OUTPUT_DIR, f"{script_name}_audio.wav")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    build_master_audio(timeline, audio_out)

    # ── Step 6: Render Video ─────────────────────────────────────────────────
    print("\n[6/6] Rendering video...")
    video_out = os.path.join(OUTPUT_DIR, f"{script_name}.mp4")
    render_video(timeline, script, audio_out, video_out, verbose=True)

    elapsed = time.time() - start
    print("\n" + "=" * 60)
    print(f"  DONE  ({elapsed:.0f}s total)")
    print(f"  Output: {video_out}")
    print("=" * 60)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--script", required=True, help="Path to script JSON")
    args = parser.parse_args()
    run_pipeline(args.script)

In [3]:
parser = argparse.ArgumentParser()

In [4]:
parser.add_argument("--script", required=True, help="Path to script JSON")

_StoreAction(option_strings=['--script'], dest='script', nargs=None, const=None, default=None, type=None, choices=None, required=True, help='Path to script JSON', metavar=None)

In [ ]:
%tb
args = parser.parse_known_args()
# C:\Users\Arunabh\Desktop\YouTube\Channel_1_01\stickman_animation\assets\scripts\episode_1.json

SystemExit: 2

usage: ipykernel_launcher.py [-h] --script SCRIPT
ipykernel_launcher.py: error: the following arguments are required: --script


SystemExit: 2

C:\Users\Arunabh\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [11]:
import os
print(os.path.exists("assets/output/episode_1_audio.wav"))  # Should print True

True


In [13]:
import subprocess, numpy as np

frame = np.zeros((720, 1280, 3), dtype=np.uint8)

cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo", "-vcodec", "rawvideo",
    "-pix_fmt", "bgr24", "-s", "1280x720", "-r", "24",
    "-i", "pipe:0",
    "-i", "assets/output/episode_1_audio.wav",  # ← same path as before
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "test_output.mp4"
]

proc = subprocess.Popen(cmd, stdin=subprocess.PIPE,
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)

try:
    for i in range(10):
        proc.stdin.write(frame.tobytes())
        print(f"Frame {i} written OK")
except BrokenPipeError:
    pass  # Expected — we just want to get to the error message below

proc.stdin.close()
_, stderr = proc.communicate()
print("Return code:", proc.returncode)
print("\n--- ffmpeg error ---")
print(stderr.decode())

Frame 0 written OK
Frame 1 written OK
Frame 2 written OK
Frame 3 written OK
Frame 4 written OK
Frame 5 written OK
Frame 6 written OK
Frame 7 written OK
Frame 8 written OK
Frame 9 written OK
Return code: 0

--- ffmpeg error ---
ffmpeg version 8.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --en